In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import math
import statsmodels.api as sm
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from mlforecast import MLForecast
import optuna
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from window_ops.rolling import rolling_mean, rolling_max, rolling_min
import os
if os.getcwd().split('\\')[-1] != 'Unconventional_Canada':
    os.chdir('..\\')
print(os.getcwd())

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

c:\Projects\AICOE International Assets\Unconventional_Canada


In [2]:
df = pd.read_csv('data/02_intermediate/data_activity_processed.csv')
df = df[:60]
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-01')
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year


In [3]:
feature = df.drop(columns=['total_opex','date','year','month'])
target = df['total_opex']

In [ ]:
corr_pearson= feature.corrwith(target, method='pearson')
corr_pearson.sort_values(ascending=False, inplace=True)
corr_pearson

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Bar(x=corr_pearson.index, y=corr_pearson, name='Pearson Correlation')
)

fig.update_layout(
    title='Pearson Correlation of OPEX Activity Dataset',
    xaxis=dict( tickfont=dict(size=13), tickangle=45),
    yaxis=dict(title='Correlation', range=[-0.25, 1], tickfont=dict(size=11)),
    autosize=False,
    width=800,
    height=500,
)

fig.show()

In [ ]:
corr_pearson.map(abs).sort_values(ascending=False)

In [ ]:
feature_pearson= corr_pearson.map(abs).sort_values().iloc[-10:].index
df_feature = pd.concat([df[['date','month','year','total_opex']],df[feature_pearson]], axis=1)
df_feature.drop(columns=['lease_rentals'], inplace=True)
df_feature['unique_id'] = 0
df_feature.tail()

In [8]:
df_train, df_test = df_feature[:54], df_feature[54:60]
h=6

In [ ]:
df_train.drop(columns=['month','year'], inplace=True)
df_train.head()

In [ ]:
df_train.columns.to_list()

In [11]:
dynamic_features = ['fluid_handling_trucking',
                    'ngl',
                    'operating_supplies',
                    'camp_field_office',
                    'carbon_tax',
                    'gas',
                    'total_prod',
                    'labor_auto',
                    'run_maintenance']

Optuna hyperparameter tuning

In [12]:
def model_tuning(actual, trial, model_class, seed):
    # Define the hyperparameters
    
    if model_class == LinearRegression:
        param = {
        }
    elif model_class == KNeighborsRegressor:
        param = {
        'n_neighbors': trial.suggest_int('n_neighbors', 1, 10),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute']),
        'leaf_size': trial.suggest_int('leaf_size', 20, 50),
        'p': trial.suggest_int('p', 1, 2),
    }
    elif model_class == DecisionTreeRegressor:
        param = {
        'random_state': seed,
        'max_depth': trial.suggest_int('max_depth', 1, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 2, 100),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 1.0),
    }
    elif model_class == RandomForestRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    elif model_class == XGBRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        }
    elif model_class == AdaBoostRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 1),
        'loss': trial.suggest_categorical('loss', ['linear', 'square', 'exponential']),
    }
    elif model_class == GradientBoostingRegressor:
        param = {
        'random_state': seed,
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    elif model_class == LGBMRegressor:
        param = {
        'random_state': seed,
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_discrete_uniform('subsample', 0.5, 0.9, 0.1),
        'colsample_bytree': trial.suggest_discrete_uniform('colsample_bytree', 0.5, 0.9, 0.1),
    }

    # Create the model
    models = [model_class(**param)]
    model = MLForecast(models=models,
                       freq='M',
                       lags=[1,2,3,4,5,6],
                       lag_transforms={
                           1: [(rolling_mean, 6), (rolling_min, 6), (rolling_max, 6)],
                       },
                       date_features=['month'],
                       num_threads=6)
    model.fit(df_train, id_col='unique_id', time_col='date', target_col='total_opex', max_horizon=6)

    # Generate forecasts
    forecast = model.predict(h=6, new_df=df_train)
    pred = forecast[['date', model_class.__name__]]
    pred.set_index('date', inplace=True)
    # Calculate the mean absolute error
    mape = mean_absolute_percentage_error(actual, pred)

    return mape

Run the optuna

In [ ]:
model_classes = [RandomForestRegressor,
                 GradientBoostingRegressor]
# model_classes = [LinearRegression, 
#                  KNeighborsRegressor, 
#                  DecisionTreeRegressor, 
#                  RandomForestRegressor, 
#                  XGBRegressor, 
#                  AdaBoostRegressor, 
#                  GradientBoostingRegressor, 
#                  LGBMRegressor]
actual = df_test[['date','total_opex']].set_index('date')
studies = {}
models = {}
model_list = []
model_general = []
for seed in np.random.randint(0,1000,10):
    for model_class in model_classes:
        study = optuna.create_study(direction='minimize')
        study.optimize(lambda trial: model_tuning(actual, trial, model_class, seed), n_trials=100)
        model_list.append(model_class.__name__ + '_' + str(seed))
        model_general.append(model_class.__name__)
        studies[model_class.__name__ + '_' + str(seed)] = study
        models[model_class.__name__ + '_' + str(seed)] = [model_class(**study.best_params)]


Prediction

In [14]:
# Create an empty DataFrame to store the predictions
result = df_test[['unique_id','date','total_opex', 'month','year']]

for model_name, model in models.items():
    model_ml = MLForecast(models=model,
                       freq='M',
                       lags=[1,2,3,4,5,6],
                       lag_transforms={
                           1: [(rolling_mean, 6), (rolling_min, 6), (rolling_max, 6)],
                       },
                       date_features=['month'],
                       num_threads=6)
    model_ml.fit(df_train, id_col='unique_id', time_col='date', target_col='total_opex', max_horizon=8)

    p = model_ml.predict(h=8, new_df=df_train)
    # p[model_name.split("_")[0]] = p[model_name.split("_")[0]].shift(-1)
    p.rename(columns={model_name.split("_")[0]: model_name}, inplace=True)
    p['month'] = p['date'].dt.month
    p['year'] = p['date'].dt.year
    p.drop(columns=['date'],inplace=True)
    result = pd.merge(result,p, on=['unique_id', 'month','year'], how='left')

In [15]:
def calculate_metrics(actual, prediction):
    mape = mean_absolute_percentage_error(actual, prediction) * 100
    mae = mean_absolute_error(actual, prediction)
    mse = mean_squared_error(actual, prediction)

    metrics_df = pd.DataFrame({
        'MAPE': [mape],
        'MAE': [mae],
        'MSE': [mse]
    })

    return metrics_df

In [ ]:
df_result = result.copy().set_index('date')
metrics_df = pd.DataFrame()
actual = df_result['total_opex']
for model_name in model_list:
    prediction = df_result[model_name]
    model_metrics = calculate_metrics(actual, prediction)
    model_metrics.index = [model_name]
    metrics_df = pd.concat([metrics_df, model_metrics]).sort_values(by='MAPE')

metrics_df.round(2)

In [ ]:
ax = metrics_df['MAPE'].sort_values().plot(kind='barh', figsize=(8, 3))  # Decrease the height of the figure
ax.set_title('MAPE by Model')
ax.set_xlabel('MAPE', fontsize=9)  # Set x label size
# Set axis label size
ax.tick_params(axis='both', labelsize=9)
ax.set_xlim([0, 20])

for p in ax.patches:
    ax.annotate(format(p.get_width(), '.2f'), 
                (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha = 'center', 
                va = 'center', 
                xytext = (15, 0), 
                fontsize= 9,
                textcoords = 'offset points')

plt.show()

In [ ]:
result_plot = df_result[model_list+['total_opex']]

ax = result_plot.plot(kind='line', figsize=(9, 4))
plt.title('Result Plot for Prediction Models')
ax.legend(loc='lower right', prop={'size': 8})  # Set legend size
ax.set_xlabel('')
plt.show()

In [ ]:
all = df[:60]
all.set_index('date', inplace=True)
all = all['total_opex']
result_plot_all = pd.concat([df_result[model_list], all], axis=1).sort_index()

ax = result_plot_all.plot(kind='line', figsize=(9, 4))
plt.title('Result Plot')
ax.legend(loc='lower right', prop={'size': 8}) 
ax.set_xlabel('')
plt.show()

Averaging randomness

In [20]:
result_mean = result.copy()
model_general = list(set(model_general))
for word in model_general:
    average = result_mean.filter(like=word).mean(axis=1)
    result_mean[word] = average

In [21]:
df_result_mean = result_mean.copy().set_index('date')
metrics_df_mean = pd.DataFrame()
actual = df_result_mean['total_opex']
for model_name in model_general:
    prediction_mean = df_result_mean[model_name]
    model_metrics_mean = calculate_metrics(actual, prediction_mean)
    model_metrics_mean.index = [model_name]
    metrics_df_mean = pd.concat([metrics_df_mean, model_metrics_mean]).sort_values(by='MAPE')

metrics_df_mean.round(2)

,MAPE,MAE,MSE
RandomForestRegressor,5.13,843700.82,9.358240e+11
GradientBoostingRegressor,5.96,967737.08,1.150716e+12


In [ ]:
ax = metrics_df_mean['MAPE'].sort_values().plot(kind='barh', figsize=(8, 3))  # Decrease the height of the figure
ax.set_title('MAPE by Model')
ax.set_xlabel('MAPE', fontsize=9)  # Set x label size
# Set axis label size
ax.tick_params(axis='both', labelsize=9)
ax.set_xlim([0, 20])

for p in ax.patches:
    ax.annotate(format(p.get_width(), '.2f'), 
                (p.get_width(), p.get_y() + p.get_height() / 2.), 
                ha = 'center', 
                va = 'center', 
                xytext = (15, 0), 
                fontsize= 9,
                textcoords = 'offset points')

plt.show()

In [ ]:
result_plot_mean = df_result_mean[model_general+['total_opex']]

ax = result_plot_mean.plot(kind='line', figsize=(9, 4))
plt.title('Result Plot for Prediction Models')
ax.legend(loc='lower right', prop={'size': 8})  # Set legend size
ax.set_xlabel('')
plt.show()

In [ ]:
all_mean = df[:60]
all_mean.set_index('date', inplace=True)
all_mean = all_mean['total_opex']
result_plot_all_mean = pd.concat([df_result_mean[model_general], all_mean], axis=1).sort_index()

ax = result_plot_all_mean.plot(kind='line', figsize=(9, 4))
plt.title('Result Plot')
ax.legend(loc='lower right', prop={'size': 8}) 
ax.set_xlabel('')
plt.show()

check date of predictions